# Step 1: Import all necessary libraries

In [ ]:
# Libraries
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)


## Step 2: Import existing bottleneck features

In [2]:
helper_bottleneck_features = pd.read_parquet('data/processed/helper_bottleneck_features.parquet')

In [3]:
helper_bottleneck_features.head()

,flight_id,taxi_start,taxi_end,pushback,sum_p10_hist_seg_time,sum_median_hist_seg_time,sum_p90_hist_seg_time,sum_seg_length_m,n_intersections_passed,TO_runway_28,TO_runway_32,TO_runway_16,TO_runway_10,TO_runway_34
0,GSW6KP_4832,2024-12-01 04:44:54+00:00,2024-12-01 04:58:20+00:00,1,NaN,NaN,NaN,711.4,4,0,1,0,0,0
1,EDW200E_4558,2024-12-01 04:58:35+00:00,2024-12-01 05:07:58+00:00,1,NaN,NaN,NaN,1190.3,7,0,1,0,0,0
2,EDW120_4456,2024-12-01 05:18:50+00:00,2024-12-01 05:25:38+00:00,1,NaN,NaN,NaN,740.8,4,0,1,0,0,0
3,EDW15T_4539,2024-12-01 05:31:07+00:00,2024-12-01 05:35:22+00:00,1,NaN,NaN,NaN,1092.5,7,0,1,0,0,0
4,EDW212R_4430,2024-12-01 05:34:06+00:00,2024-12-01 05:46:14+00:00,1,NaN,NaN,NaN,1911.9,11,0,1,0,0,0


## Step 3: Import processed trajectories

In [4]:
trajs_processed = pd.read_parquet('data/processed/trajs_processed.parquet')

In [5]:
trajs_processed.head()

,timestamp,icao24,latitude,longitude,groundspeed,track,vertical_rate,callsign,onground,alert,spi,squawk,altitude,geoaltitude,lastcontact,serials,hour,flight_id,track_unwrapped,cumdist,compute_gs,compute_track,registration,typecode,cut_runway,x,y,groundspeed_xy,track_xy,latitude_kalman,longitude_kalman,groundspeed_kalman,track_kalman,pushback
0,2024-12-03 19:27:23+00:00,501e43,47.454506,8.572392,<NA>,NaN,<NA>,9AJIM,True,False,False,1000,1200.0,<NA>,1733254042.608,[-1408231728],2024-12-03 19:00:00+00:00,9AJIM_8350,NaN,NaN,6.656234,281.121521,9A-JIM,C525,28,1835.107443,-394.409784,10.289896,288.756018,47.454498,8.572272,5.635465,298.075857,0
1,2024-12-03 19:27:24+00:00,501e43,47.454525,8.572311,<NA>,NaN,<NA>,9AJIM,True,False,False,1000,1200.0,<NA>,1733254043.929,[-1408231728],2024-12-03 19:00:00+00:00,9AJIM_8350,NaN,NaN,6.656234,327.148395,9A-JIM,C525,28,1829.065901,-392.298275,9.573292,288.535804,47.454504,8.572235,5.830707,303.050765,0
2,2024-12-03 19:27:25+00:00,501e43,47.454537,8.572259,<NA>,NaN,<NA>,9AJIM,True,False,False,1000,1200.0,<NA>,1733254043.929,[-1408231728],2024-12-03 19:00:00+00:00,9AJIM_8350,NaN,NaN,6.656234,327.148395,9A-JIM,C525,28,1825.082500,-391.005600,8.543856,295.514329,47.454516,8.572201,6.047056,307.521679,0
3,2024-12-03 19:27:26+00:00,501e43,47.454548,8.572206,<NA>,NaN,<NA>,9AJIM,True,False,False,1000,1200.0,<NA>,1733254045.899,[-1408231728],2024-12-03 19:00:00+00:00,9AJIM_8350,NaN,NaN,6.656234,327.148395,9A-JIM,C525,28,1821.099101,-389.712922,6.975231,305.511766,47.454536,8.572171,6.373721,315.379196,0
4,2024-12-03 19:27:27+00:00,501e43,47.454574,8.572181,<NA>,NaN,<NA>,9AJIM,True,False,False,1000,1200.0,<NA>,1733254045.899,[-1408231728],2024-12-03 19:00:00+00:00,9AJIM_8350,NaN,NaN,6.656234,327.148395,9A-JIM,C525,28,1819.240662,-386.836846,6.694868,316.065216,47.454562,8.572144,6.676477,321.610645,0


# Step 4: Import processed meteorological data

In [6]:
meteo_processed = pd.read_parquet('data/processed/meteo_processed.parquet')

## Step 5: Feature Engineering

In the following steps, additional features based on the processed trajectories are being created and then merged with the existing bottleneck features.

### Step 5.1: Operational Features

First, we create a features dataframe and define the target variable (taxi out time) for each flight_id.

In [7]:
features_df = helper_bottleneck_features[
    ["flight_id", "taxi_start", "taxi_end"]
].copy()

features_df["taxi_time_min"] = (
    features_df["taxi_end"] - features_df["taxi_start"]
).dt.total_seconds() / 60

Before defining a filtering threshold, the distribution of taxi-out times is inspected using summary statistics and lower quantiles to understand the realistic range of values.

In [8]:
def inspect_taxi_time_distribution(features_df):

    print("Summary statistics:")
    print(features_df["taxi_time_min"].describe())

    print("\nLower quantiles:")
    print(
        features_df["taxi_time_min"].quantile(
            [0.01, 0.02, 0.05, 0.10]
        )
    )


inspect_taxi_time_distribution(features_df)

Summary statistics:
count     62272.000000
mean        279.206508
std        6821.514846
min           0.050000
25%           6.283333
50%           9.366667
75%          12.383333
max      348546.150000
Name: taxi_time_min, dtype: float64

Lower quantiles:
0.01    0.683333
0.02    1.300000
0.05    2.233333
0.10    3.266667
Name: taxi_time_min, dtype: float64


A configurable minimum and maximum taxi-out time parameter is defined to allow flexible adjustment of the filtering threshold as the dataset grows or data characteristics change.


 <font color='red'> CHANGE THE PARAMETER IN THE CELL BELOW: </font>

In [9]:
MINIMUM_TAXI_TIME = 1
MAXIMUM_TAXI_TIME = 60

Flights with taxi-out times below the defined minimum threshold are excluded from further analysis to remove incomplete or operationally implausible trajectories.

In [10]:
features_df = features_df[
    (features_df["taxi_time_min"] >= MINIMUM_TAXI_TIME) &
    (features_df["taxi_time_min"] <= MAXIMUM_TAXI_TIME)
]

Adding aircraft typecode as a feature and setting datatype as "category"

In [11]:
typecode_lookup = (
    trajs_processed[["flight_id", "typecode"]]
    .dropna(subset=["typecode"])
    .drop_duplicates()
)

features_df = features_df.merge(
    typecode_lookup,
    on="flight_id",
    how="left"
)

features_df["typecode"] = features_df["typecode"].astype("category")

The queue size feature is calculated within a rolling one-hour lookback window. For each flight, only aircraft that started taxiing within the previous hour and were still taxiing at the current flight's taxi_start time are counted. This keeps the feature operationally realistic, prediction-safe, and computationally scalable for larger datasets.

In [12]:
WINDOW = pd.Timedelta(hours=1)

features_df = features_df.sort_values("taxi_start")

starts = features_df["taxi_start"]
ends = features_df["taxi_end"]

queue_size_last_hour = []

for i, t in enumerate(starts):
    n_active = (
        (starts >= t - WINDOW)
        & (starts < t)
        & (ends > t)
    ).sum()

    queue_size_last_hour.append(n_active)

features_df["queue_size"] = queue_size_last_hour

Implementing the average taxi out time of the past hour as a feature.

In [13]:
WINDOW = pd.Timedelta(hours=1)

features_df = features_df.sort_values("taxi_start")

starts = features_df["taxi_start"]
ends = features_df["taxi_end"]
taxi_times = features_df["taxi_time_min"]

avg_taxi_out_1h = []

for i, t in enumerate(starts):

    mask = (
        (ends >= t - WINDOW)
        & (ends < t)
    )

    if mask.any():
        avg = taxi_times[mask].mean()
    else:
        avg = None

    avg_taxi_out_1h.append(avg)

features_df["avg_taxi_out_time_1h"] = avg_taxi_out_1h

Implementing bottleneck features

In [14]:
cols_to_merge = [
    "flight_id",
    "n_intersections_passed",
    "sum_seg_length_m",
    "sum_p10_hist_seg_time",
    "sum_median_hist_seg_time",
    "sum_p90_hist_seg_time",
    "TO_runway_28",
    "TO_runway_32",
    "TO_runway_16",
    "TO_runway_10",
    "TO_runway_34",
    "pushback",
]

features_df = features_df.merge(
    helper_bottleneck_features[cols_to_merge],
    on="flight_id",
    how="left",
)

### Step 5.2 Temporal Features

Defining a function for cyclic encoding

In [15]:
def sin_cos(x,T):
    return np.sin(2 * np.pi * x / T), np.cos(2 * np.pi * x / T)

Splitting up information from the 'taxi_start' column into month, day of week and minute of day

In [16]:
features_df["month"] = features_df["taxi_start"].dt.month

features_df["day_of_week"] = features_df["taxi_start"].dt.dayofweek
# Monday = 0, Sunday = 6

features_df["minute_of_day"] = (
    features_df["taxi_start"].dt.hour * 60
    + features_df["taxi_start"].dt.minute
)

Encoding month, day of week and minute of day as sin and cos

In [ ]:
features_df["minute_of_day_sin"], features_df["minute_of_day_cos"] = sin_cos(
    features_df["minute_of_day"], 24 * 60
)

### Step 5.3: Meteorological Features

Implementing temperature and spread as  features

In [ ]:
meteo_temp = (
    meteo_processed[
        ["reference_timestamp", "tre200s0"]
    ]
    .rename(
        columns={
            "tre200s0": "temp_at_taxi_start"
        }
    )
)

features_df = pd.merge_asof(
    features_df.sort_values("taxi_start"),
    meteo_temp.sort_values("reference_timestamp"),
    left_on="taxi_start",
    right_on="reference_timestamp",
    direction="backward"
)

features_df = features_df.drop(columns="reference_timestamp")

In [19]:
features_df.head(6)

,flight_id,taxi_start,taxi_end,taxi_time_min,typecode,queue_size,avg_taxi_out_time_1h,n_intersections_passed,sum_seg_length_m,sum_p10_hist_seg_time,sum_median_hist_seg_time,sum_p90_hist_seg_time,TO_runway_28,TO_runway_32,TO_runway_16,TO_runway_10,TO_runway_34,pushback,month,day_of_week,minute_of_day,month_sin,month_cos,day_of_week_sin,day_of_week_cos,minute_of_day_sin,minute_of_day_cos,temp_at_taxi_start,spread_at_taxi_start
0,GSW6KP_4832,2024-12-01 04:44:54+00:00,2024-12-01 04:58:20+00:00,13.433333,A320,0,NaN,4,711.4,NaN,NaN,NaN,0,1,0,0,0,1,12,6,284,-2.449294e-16,1.0,-0.781831,0.62349,0.945519,0.325568,0.3,2.4
1,EDW200E_4558,2024-12-01 04:58:35+00:00,2024-12-01 05:07:58+00:00,9.383333,A320,0,13.433333,7,1190.3,NaN,NaN,NaN,0,1,0,0,0,1,12,6,298,-2.449294e-16,1.0,-0.781831,0.62349,0.963630,0.267238,0.3,2.4
2,EDW120_4456,2024-12-01 05:18:50+00:00,2024-12-01 05:25:38+00:00,6.800000,A320,0,11.408333,4,740.8,NaN,NaN,NaN,0,1,0,0,0,1,12,6,318,-2.449294e-16,1.0,-0.781831,0.62349,0.983255,0.182236,0.2,2.4
3,EDW15T_4539,2024-12-01 05:31:07+00:00,2024-12-01 05:35:22+00:00,4.250000,A320,0,9.872222,7,1092.5,NaN,NaN,NaN,0,1,0,0,0,1,12,6,331,-2.449294e-16,1.0,-0.781831,0.62349,0.992005,0.126199,0.3,2.3
4,EDW212R_4430,2024-12-01 05:34:06+00:00,2024-12-01 05:46:14+00:00,12.133333,A320,1,9.872222,11,1911.9,NaN,NaN,NaN,0,1,0,0,0,1,12,6,334,-2.449294e-16,1.0,-0.781831,0.62349,0.993572,0.113203,0.3,2.3
5,EDW214A_4400,2024-12-01 05:51:31+00:00,2024-12-01 06:03:00+00:00,11.483333,A320,0,9.200000,9,1450.9,NaN,NaN,NaN,0,1,0,0,0,1,12,6,351,-2.449294e-16,1.0,-0.781831,0.62349,0.999229,0.039260,0.2,2.3


Export features as parquet file

In [20]:
output_path = "data/processed/features.parquet"
 
# Save dataframe to pkl.
features_df.to_parquet(output_path)
 
print(f"File \"features\" saved to: {output_path}")
 

File "features" saved to: data/processed/features.parquet
